> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 03 — low-dimensional shared genetic/environmental structure

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str);train.columns=train.columns.str.replace('regressedlr_','',regex=False)
store=vigor.open_store(ANALYSIS_STORE,mode='a',lazy=False)
try:
    cov=store.open_dataset('covariance/GE_full',lazy=False)
except Exception as exc:
    raise RuntimeError('Run project notebook 02 first so covariance/GE_full exists.') from exc
G=np.asarray(cov['G'],float); E=np.asarray(cov['E'],float)
if G.ndim==3:G=G.sum(axis=0)
print('Loaded covariance:',G.shape,E.shape)


In [ ]:
rank=min(10,len(train.columns))
dec_fa=vigor.decompose_GE(G=G,E=E,rank_G=rank,rank_E=rank,method_G='fa',method_E='fa')
dec_pca=vigor.decompose_GE(G=G,E=E,rank_G=rank,rank_E=rank,method_G='pca',method_E='pca')
summary=pd.DataFrame([
    {'method':'FA','G_explained_trace':dec_fa.attrs['G']['explained_trace'],'E_explained_trace':dec_fa.attrs['E']['explained_trace']},
    {'method':'PCA','G_explained_trace':dec_pca.attrs['G']['explained_trace'],'E_explained_trace':dec_pca.attrs['E']['explained_trace']},
])
display(summary)
summary.to_csv(RESULTS/'GE_factor_decomposition_summary.csv',index=False)


In [ ]:
# Store both G and E loadings in one labeled decomposition group.
L=np.stack([dec_fa['G']['loadings'],dec_fa['E']['loadings']])
psi=np.stack([dec_fa['G']['specific_var'],dec_fa['E']['specific_var']])
store.write_decomposition(
    loadings=L,specific_variance=psi,
    component_names=['G','E'],trait_names=train.columns,
    factor_names=[f'F{i+1}' for i in range(L.shape[2])],
    group='decomposition/GE_full_FA10',mode='a',
)
print('Persisted decomposition/GE_full_FA10')


Use FA for the biological shared-vs-specific variance interpretation and PCA as an orthogonal descriptive comparator. For cross-fit/bootstrap interpretation, align loadings before naming axes.